In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import os
from sklearn.metrics import classification_report, confusion_matrix
import pickle

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM, SpatialDropout1D, Lambda, Bidirectional, GRU
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Concatenate, Dropout, Dense, Lambda
from tensorflow.keras.metrics import Precision, Recall


from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from nltk.tokenize import word_tokenize

from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

from datasets import Dataset, DatasetDict

BASE_PATH = os.path.dirname(os.getcwd())
DATASET_PATH = BASE_PATH+"/datasets"
GLOVE_FILE = DATASET_PATH+"/GLOVE/gloveVocab.txt"
WORD2VECFILE = DATASET_PATH+"/GLOVE/word2vecVocab.txt"
FAST_TEXT_FILE = DATASET_PATH+"/FAST_TEXT/fastTextVocab.vec"

with open(f"{BASE_PATH}/datasets/material/readyDataset.pkl", "rb") as f : 
    df = pickle.load(f)
print("LOADED")

LOADED


In [ ]:
glove2word2vec(GLOVE_FILE, WORD2VECFILE)
print("Word2Vec File was Made Already !!")

C:\Users\Reynaldi\AppData\Local\Temp\ipykernel_11928\1159042987.py:1: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(GLOVE_FILE, WORD2VECFILE)


Word2Vec File was Made Already !!


In [ ]:
fastTextModel = KeyedVectors.load_word2vec_format(FAST_TEXT_FILE)
gloveModel = KeyedVectors.load_word2vec_format(WORD2VECFILE)

In [13]:
with open(f"{BASE_PATH}/models/gloveModel.pkl","wb") as f : 
    pickle.dump(gloveModel, f)
with open(f"{BASE_PATH}/models/fastTextModel.pkl","wb") as f : 
    pickle.dump(fastTextModel, f)
print("Model was Succesfully Saved !!")

Model was Succesfully Saved !!


In [2]:
with open(f"{BASE_PATH}/models/gloveModel.pkl","rb") as f : 
    gloveModel = pickle.load(f)
with open(f"{BASE_PATH}/models/fastTextModel.pkl","rb") as f : 
    fastTextModel = pickle.load(f)
print("Model was Succesfully Loaded !!")

Model was Succesfully Loaded !!


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

labelEncoder = LabelEncoder()
LABEL = labelEncoder.fit_transform(df["label"])
train_data, test_data, train_label, test_label = train_test_split(df["Cleaned_Text"], LABEL, test_size = 0.2, random_state=42, stratify=LABEL)

In [4]:
def embeddingTheSentence(sentence : str, maks_number = 100) :
    sentence = sentence.strip().split(" ")
    result = []
    for word in sentence : 
        result.append([])
        try : 
            resultGlove = gloveModel[word]
        except : 
            resultGlove = [0] * 300
        try : 
            resultFastText = fastTextModel[word]
        except : 
            resultFastText = [0] * 300
        result[-1].extend(resultGlove)
        result[-1].extend(resultFastText)
    if len(result) > 100 : 
        result = result[:maks_number]
    else : 
        berapa_0 = maks_number - len(result)
        list_0 = [[0] * 600] * berapa_0
        result.extend(list_0)
    hasil = np.array(result)
    return hasil

In [5]:
MAKS_NUMBER = 100

train_array = []
for sentence in train_data.values : 
    train_array.append(embeddingTheSentence(sentence))

test_array = []
for sentence in test_data.values : 
    test_array.append(embeddingTheSentence(sentence))

train_array = np.array(train_array)
test_array = np.array(test_array)

In [6]:
with open(f"{BASE_PATH}/datasets/processed/train_embeddings.pkl","wb") as f : 
    pickle.dump(train_array, f)
with open(f"{BASE_PATH}/datasets/processed/test_embeddings.pkl","wb") as f : 
    pickle.dump(test_array, f)
with open(f"{BASE_PATH}/datasets/processed/train_labels.pkl","wb") as f : 
    pickle.dump(train_label, f)
with open(f"{BASE_PATH}/datasets/processed/test_labels.pkl","wb") as f : 
    pickle.dump(test_label, f)
    
print("Array was Succesfully Saved !!")

Array was Succesfully Saved !!


In [ ]:
with open(f"{BASE_PATH}/datasets/processed/train_embeddings.pkl","rb") as f : 
    train_array = pickle.load(f)
with open(f"{BASE_PATH}/datasets/processed/test_embeddings.pkl","rb") as f : 
    test_array = pickle.load(f)
with open(f"{BASE_PATH}/datasets/processed/train_labels.pkl","rb") as f : 
    train_label = pickle.load(f)
with open(f"{BASE_PATH}/datasets/processed/test_labels.pkl","rb") as f : 
    test_label = pickle.load(f)
print("Array was Succesfully Loaded !!")

Array was Succesfully Loaded !!


### 1. Bidirectional GRU Architecture (Combination GloVe + FastText)

In [13]:
gru_units_1 = 128
gru_units_2 = 64

model = Sequential()
model.add(Input(shape = (100, 600)))
model.add(SpatialDropout1D(0.2))

# Bidirectional GRU
model.add(Bidirectional(GRU(gru_units_1, return_sequences=True)))

# Bidirectional GRU / LSTM kedua
model.add(Bidirectional(GRU(gru_units_2, return_sequences=True)))

# Global pooling → concatenate average + max
def global_concat(x):
    avg_pool = K.mean(x, axis=1)
    max_pool = K.max(x, axis=1)
    return K.concatenate([avg_pool, max_pool], axis=-1)

model.add(Lambda(global_concat))

# Dropout sebelum dense
model.add(Dropout(0.2))
model.add(Dense(6, activation= "softmax"))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spatial_dropout1d_1             │ (None, 100, 600)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 256)       │       560,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 100, 128)       │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 685,830 (2.62 MB)

 Trainable params: 685,830 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
class_weight = compute_class_weight(class_weight="balanced", classes = np.unique(train_label), y = train_label)
class_weight = {i:float(t) for i, t in enumerate(class_weight)}
class_weight

{0: 0.5749604714675866,
 1: 0.4930054846860171,
 2: 2.030972328002031,
 3: 1.2305799107829565,
 4: 1.4049877063575693,
 5: 4.63768115942029}

In [ ]:
adam = Adam(learning_rate=0.0005)
model.compile(  
    optimizer=adam,  
    loss="sparse_categorical_crossentropy",  
    metrics=["sparse_categorical_accuracy"]
)

In [16]:
import time

start = time.time()

history = model.fit(
          x=train_array,
          y=train_label,
          batch_size=64,
          epochs=100, verbose=1, 
          callbacks=[EarlyStopping(monitor = "val_loss", patience = 30)], 
          validation_split=0.2, 
          class_weight=class_weight)

print(f"Time Needed : {time.time() - start}")

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 25s 115ms/step - loss: 0.9651 - sparse_categorical_accuracy: 0.6128 - val_loss: 0.4061 - val_sparse_categorical_accuracy: 0.8637
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 24s 120ms/step - loss: 0.2797 - sparse_categorical_accuracy: 0.8866 - val_loss: 0.2504 - val_sparse_categorical_accuracy: 0.9094
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 26s 131ms/step - loss: 0.1962 - sparse_categorical_accuracy: 0.9176 - val_loss: 0.2048 - val_sparse_categorical_accuracy: 0.9228
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 26s 129ms/step - loss: 0.1702 - sparse_categorical_accuracy: 0.9280 - val_loss: 0.1867 - val_sparse_categorical_accuracy: 0.9231
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 25s 125ms/step - loss: 0.1464 - sparse_categorical_accuracy: 0.9341 - val_loss: 0.1856 - val_sparse_categorical_accuracy: 0.9294
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 26s 128ms/step - loss: 0.1390 - sparse_categorical_accuracy: 0.9378 - val_loss: 0.1670 - val_sparse_categoric

In [17]:
from sklearn.metrics import classification_report

predicted = model.predict(test_array)
predicted = np.argmax(predicted, axis = -1)
predicted = np.int8(predicted)
print(classification_report(y_true = test_label, y_pred = predicted, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step
              precision    recall  f1-score   support

           0      0.976     0.966     0.971      1159
           1      0.948     0.945     0.947      1352
           2      0.858     0.829     0.843       328
           3      0.902     0.952     0.926       542
           4      0.916     0.895     0.905       475
           5      0.776     0.819     0.797       144

    accuracy                          0.932      4000
   macro avg      0.896     0.901     0.898      4000
weighted avg      0.933     0.932     0.932      4000



### 2. Bidirectional LSTM Architecture (Combination GloVe + FastText)

In [18]:
lstm_units_1 = 128
lstm_units_2 = 64

model = Sequential()
model.add(Input(shape = (100, 600)))
model.add(SpatialDropout1D(0.2))

# Bidirectional GRU
model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True, recurrent_dropout=0.2)))

# Bidirectional GRU / LSTM kedua
model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))

# Global pooling → concatenate average + max
def global_concat(x):
    avg_pool = K.mean(x, axis=1)
    max_pool = K.max(x, axis=1)
    return K.concatenate([avg_pool, max_pool], axis=-1)

model.add(Lambda(global_concat))

# Dropout sebelum dense
model.add(Dropout(0.2))
model.add(Dense(6, activation= "softmax"))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spatial_dropout1d_2             │ (None, 100, 600)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 100, 256)       │       746,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 100, 128)       │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 912,390 (3.48 MB)

 Trainable params: 912,390 (3.48 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
class_weight = compute_class_weight(class_weight="balanced", classes = np.unique(train_label), y = train_label)
class_weight = {i:float(t) for i, t in enumerate(class_weight)}
class_weight

{0: 0.5749604714675866,
 1: 0.4930054846860171,
 2: 2.030972328002031,
 3: 1.2305799107829565,
 4: 1.4049877063575693,
 5: 4.63768115942029}

In [20]:
adam = Adam(learning_rate=0.0005)
model.compile(  
    optimizer=adam,  
    loss="sparse_categorical_crossentropy",  
    metrics=["sparse_categorical_accuracy"]
)  

start = time.time()
model.fit(x=train_array,
          y=train_label,
          batch_size=64,
          epochs=100, verbose=1, 
          callbacks=[EarlyStopping(monitor = "val_loss", patience = 30)], 
          validation_split=0.2, 
          class_weight=class_weight)
print(f"Time Needed : {time.time() - start}")

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 43s 200ms/step - loss: 1.4134 - sparse_categorical_accuracy: 0.3865 - val_loss: 0.8768 - val_sparse_categorical_accuracy: 0.6956
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 38s 190ms/step - loss: 0.6369 - sparse_categorical_accuracy: 0.7434 - val_loss: 0.5203 - val_sparse_categorical_accuracy: 0.8247
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 40s 201ms/step - loss: 0.4019 - sparse_categorical_accuracy: 0.8393 - val_loss: 0.4351 - val_sparse_categorical_accuracy: 0.8447
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 39s 197ms/step - loss: 0.3034 - sparse_categorical_accuracy: 0.8748 - val_loss: 0.2969 - val_sparse_categorical_accuracy: 0.8928
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 40s 200ms/step - loss: 0.2481 - sparse_categorical_accuracy: 0.8952 - val_loss: 0.2679 - val_sparse_categorical_accuracy: 0.8978
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 40s 200ms/step - loss: 0.2132 - sparse_categorical_accuracy: 0.9062 - val_loss: 0.2766 - val_sparse_categoric

In [21]:
predicted = model.predict(test_array)
predicted = np.argmax(predicted, axis = -1)
predicted = np.int8(predicted)
print(classification_report(y_true = test_label, y_pred = predicted, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step
              precision    recall  f1-score   support

           0      0.971     0.959     0.965      1159
           1      0.950     0.919     0.935      1352
           2      0.811     0.835     0.823       328
           3      0.922     0.915     0.919       542
           4      0.874     0.903     0.888       475
           5      0.706     0.882     0.784       144

    accuracy                          0.920      4000
   macro avg      0.872     0.902     0.886      4000
weighted avg      0.923     0.920     0.921      4000



### 3. Support Vector Machine Linear (Combination GloVe + FastText)

In [22]:
flatten_data = train_array.reshape(train_array.shape[0], -1)
flatten_data.shape

(16000, 60000)

In [23]:
class_weight = compute_class_weight(class_weight="balanced", classes = np.unique(train_label), y = train_label)
class_weight = {i:float(t) for i, t in enumerate(class_weight)}

from sklearn.svm import LinearSVC
model = LinearSVC(class_weight = class_weight, random_state = 42, verbose = 1)

start = time.time()
model.fit(flatten_data, train_label)
print(f"Time Needed : {time.time() - start}")

[LibLinear]Time Needed : 943.1832060813904


d:\Conda\envs\tensorflow_support\lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [24]:
flatten_test_data = test_array.reshape(test_array.shape[0], -1)
flatten_test_data.shape

(4000, 60000)

In [25]:
predicted = model.predict(flatten_test_data)
print(classification_report(y_true = test_label, y_pred = predicted, digits = 3))

              precision    recall  f1-score   support

           0      0.621     0.617     0.619      1159
           1      0.696     0.675     0.685      1352
           2      0.384     0.375     0.380       328
           3      0.467     0.480     0.473       542
           4      0.469     0.499     0.484       475
           5      0.252     0.271     0.261       144

    accuracy                          0.572      4000
   macro avg      0.482     0.486     0.484      4000
weighted avg      0.575     0.572     0.573      4000



### 4. Bidirectional GRU Architecture (GloVe Only)

In [40]:
def embeddingTheSentence(sentence : str, maks_number = 100) :
    sentence = sentence.strip().split(" ")
    result = []
    for word in sentence : 
        result.append([])
        try : 
            resultGlove = gloveModel[word]
        except : 
            resultGlove = [0] * 300
        result[-1].extend(resultGlove)
    if len(result) > 100 : 
        result = result[:maks_number]
    else : 
        berapa_0 = maks_number - len(result)
        list_0 = [[0] * 300] * berapa_0
        result.extend(list_0)
    hasil = np.array(result)
    return hasil

In [41]:
MAKS_NUMBER = 100

train_array = []
for sentence in train_data.values : 
    train_array.append(embeddingTheSentence(sentence))

test_array = []
for sentence in test_data.values : 
    test_array.append(embeddingTheSentence(sentence))

train_array = np.array(train_array)
test_array = np.array(test_array)

In [42]:
gru_units_1 = 128
gru_units_2 = 64

model = Sequential()
model.add(Input(shape = (100, 300)))
model.add(SpatialDropout1D(0.2))

# Bidirectional GRU
model.add(Bidirectional(GRU(gru_units_1, return_sequences=True)))

# Bidirectional GRU / LSTM kedua
model.add(Bidirectional(GRU(gru_units_2, return_sequences=True)))

# Global pooling → concatenate average + max
def global_concat(x):
    avg_pool = K.mean(x, axis=1)
    max_pool = K.max(x, axis=1)
    return K.concatenate([avg_pool, max_pool], axis=-1)

model.add(Lambda(global_concat))

# Dropout sebelum dense
model.add(Dropout(0.2))
model.add(Dense(6, activation= "softmax"))
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spatial_dropout1d_5             │ (None, 100, 300)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 100, 256)       │       330,240 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ (None, 100, 128)       │       123,648 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_5 (Lambda)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 455,430 (1.74 MB)

 Trainable params: 455,430 (1.74 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
class_weight = compute_class_weight(class_weight="balanced", classes = np.unique(train_label), y = train_label)
class_weight = {i:float(t) for i, t in enumerate(class_weight)}
class_weight

{0: 0.5749604714675866,
 1: 0.4930054846860171,
 2: 2.030972328002031,
 3: 1.2305799107829565,
 4: 1.4049877063575693,
 5: 4.63768115942029}

In [44]:
adam = Adam(learning_rate=0.0005)
model.compile(  
    optimizer=adam,  
    loss="sparse_categorical_crossentropy",  
    metrics=["sparse_categorical_accuracy"]
)

In [45]:
import time

start = time.time()

history = model.fit(x=train_array,
          y=train_label,
          batch_size=64,
          epochs=100, verbose=1, 
          callbacks=[EarlyStopping(monitor = "val_loss", patience = 30)], 
          validation_split=0.2, 
          class_weight=class_weight)

print(f"Time Needed : {time.time() - start}")

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 24s 105ms/step - loss: 1.3941 - sparse_categorical_accuracy: 0.4209 - val_loss: 0.8927 - val_sparse_categorical_accuracy: 0.6653
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 100ms/step - loss: 0.5882 - sparse_categorical_accuracy: 0.7601 - val_loss: 0.4272 - val_sparse_categorical_accuracy: 0.8450
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 99ms/step - loss: 0.3785 - sparse_categorical_accuracy: 0.8467 - val_loss: 0.3156 - val_sparse_categorical_accuracy: 0.8784
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - loss: 0.2980 - sparse_categorical_accuracy: 0.8775 - val_loss: 0.2759 - val_sparse_categorical_accuracy: 0.8925
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 96ms/step - loss: 0.2549 - sparse_categorical_accuracy: 0.8923 - val_loss: 0.2499 - val_sparse_categorical_accuracy: 0.9000
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 101ms/step - loss: 0.2237 - sparse_categorical_accuracy: 0.9013 - val_loss: 0.2341 - val_sparse_categorical_

In [46]:
predicted = model.predict(test_array)
predicted = np.argmax(predicted, axis = -1)
predicted = np.int8(predicted)
print(classification_report(y_true = test_label, y_pred = predicted, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step
              precision    recall  f1-score   support

           0      0.974     0.965     0.970      1159
           1      0.955     0.933     0.944      1352
           2      0.798     0.854     0.825       328
           3      0.912     0.935     0.923       542
           4      0.903     0.903     0.903       475
           5      0.805     0.833     0.819       144

    accuracy                          0.929      4000
   macro avg      0.891     0.904     0.897      4000
weighted avg      0.930     0.929     0.929      4000



### 5. Bidirectional GRU Architecture (FastText Only)

In [47]:
def embeddingTheSentence(sentence : str, maks_number = 100) :
    sentence = sentence.strip().split(" ")
    result = []
    for word in sentence : 
        result.append([])
        try : 
            resultFastText = fastTextModel[word]
        except : 
            resultFastText = [0] * 300
        result[-1].extend(resultFastText)
    if len(result) > 100 : 
        result = result[:maks_number]
    else : 
        berapa_0 = maks_number - len(result)
        list_0 = [[0] * 300] * berapa_0
        result.extend(list_0)
    hasil = np.array(result)
    return hasil

In [48]:
MAKS_NUMBER = 100

train_array = []
for sentence in train_data.values : 
    train_array.append(embeddingTheSentence(sentence))

test_array = []
for sentence in test_data.values : 
    test_array.append(embeddingTheSentence(sentence))

train_array = np.array(train_array)
test_array = np.array(test_array)

In [49]:
gru_units_1 = 128
gru_units_2 = 64

model = Sequential()
model.add(Input(shape = (100, 300)))
model.add(SpatialDropout1D(0.2))

# Bidirectional GRU
model.add(Bidirectional(GRU(gru_units_1, return_sequences=True)))

# Bidirectional GRU / LSTM kedua
model.add(Bidirectional(GRU(gru_units_2, return_sequences=True)))

# Global pooling → concatenate average + max
def global_concat(x):
    avg_pool = K.mean(x, axis=1)
    max_pool = K.max(x, axis=1)
    return K.concatenate([avg_pool, max_pool], axis=-1)

model.add(Lambda(global_concat))

# Dropout sebelum dense
model.add(Dropout(0.2))
model.add(Dense(6, activation= "softmax"))
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spatial_dropout1d_6             │ (None, 100, 300)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ (None, 100, 256)       │       330,240 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ (None, 100, 128)       │       123,648 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_6 (Lambda)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 455,430 (1.74 MB)

 Trainable params: 455,430 (1.74 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
class_weight = compute_class_weight(class_weight="balanced", classes = np.unique(train_label), y = train_label)
class_weight = {i:float(t) for i, t in enumerate(class_weight)}
class_weight

{0: 0.5749604714675866,
 1: 0.4930054846860171,
 2: 2.030972328002031,
 3: 1.2305799107829565,
 4: 1.4049877063575693,
 5: 4.63768115942029}

In [51]:
adam = Adam(learning_rate=0.0005)
model.compile(  
    optimizer=adam,  
    loss="sparse_categorical_crossentropy",  
    metrics=["sparse_categorical_accuracy"]
)

In [52]:
import time

start = time.time()

history = model.fit(x=train_array,
          y=train_label,
          batch_size=64,
          epochs=100, verbose=1, 
          callbacks=[EarlyStopping(monitor = "val_loss", patience = 30)], 
          validation_split=0.2, 
          class_weight=class_weight)

print(f"Time Needed : {time.time() - start}")

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 23s 96ms/step - loss: 1.4780 - sparse_categorical_accuracy: 0.3975 - val_loss: 0.7758 - val_sparse_categorical_accuracy: 0.7969
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - loss: 0.5569 - sparse_categorical_accuracy: 0.7885 - val_loss: 0.4274 - val_sparse_categorical_accuracy: 0.8584
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - loss: 0.3457 - sparse_categorical_accuracy: 0.8610 - val_loss: 0.2694 - val_sparse_categorical_accuracy: 0.9034
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 95ms/step - loss: 0.2762 - sparse_categorical_accuracy: 0.8884 - val_loss: 0.2197 - val_sparse_categorical_accuracy: 0.9122
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 99ms/step - loss: 0.2417 - sparse_categorical_accuracy: 0.8998 - val_loss: 0.2247 - val_sparse_categorical_accuracy: 0.9153
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - loss: 0.2195 - sparse_categorical_accuracy: 0.9099 - val_loss: 0.1953 - val_sparse_categorical_acc

In [53]:
predicted = model.predict(test_array)
predicted = np.argmax(predicted, axis = -1)
predicted = np.int8(predicted)
print(classification_report(y_true = test_label, y_pred = predicted, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step
              precision    recall  f1-score   support

           0      0.979     0.956     0.967      1159
           1      0.980     0.904     0.940      1352
           2      0.761     0.942     0.842       328
           3      0.899     0.956     0.927       542
           4      0.887     0.888     0.887       475
           5      0.748     0.847     0.795       144

    accuracy                          0.925      4000
   macro avg      0.876     0.916     0.893      4000
weighted avg      0.931     0.925     0.927      4000

